# **Maestría en Inteligencia Artificial Aplicada**

## Curso: **Análisis de grandes volumenes de datos**

### Tecnológico de Monterrey

### Prof Dr. Iván Olmos Pineda

## Actividad Semana 6

### **Aprendizaje supervisado y no supervisado**

#### **Nombre y matrícula**

*   Emmanuel Merida Toledo A01795858


In [111]:
# imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum
from pyspark.sql.functions import hour, dayofweek
from pyspark.sql.functions import lead, when

from pyspark.ml.classification import GBTClassifier
from pyspark.ml.clustering import KMeans

from pyspark.sql.window import Window
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler

from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.evaluation import ClusteringEvaluator

In [112]:
spark = SparkSession.builder.getOrCreate()

# **Pregunta - 1:** Introducción teórica

**Aprendizaje supervisado** es una técnica en el cual los modelos se entrenan utilizando un conjunto de datos que incluye tanto las características como las salidas esperadas (etiquetas). El objetivo es que el modelo aprenda una función que pueda predecir correctamente la etiqueta para nuevas instancias.

Algoritmos más representativos:

*   **Árboles de decisión (DecisionTreeClassifier):** generan reglas de decisión a partir de los atributos para clasificar los datos.

*   **Random Forest:** ensamble de múltiples árboles para mejorar precisión y reducir sobreajuste.

*   **Multilayer Perceptron (MLPClassifier):** redes neuronales multicapa adecuadas para problemas no lineales.

*   **GBTClassifier (Gradient Boosted Trees):** modelo basado en árboles secuenciales que mejora iterativamente los errores.

**Aprendizaje no supervisado**, en contraste, se utiliza cuando los datos no están etiquetados. Su objetivo principal es encontrar estructuras ocultas o patrones en los datos. Ejemplos típicos:

*   **K-means:** particiona los datos en grupos (clusters) minimizando la distancia intra-cluster.

*   **GaussianMixture:** modelo probabilístico basado en una combinación de distribuciones normales.

*   **PIC (Power Iteration Clustering):** método basado en grafos para encontrar agrupamientos eficientemente.

# **Pregunta - 2:** Selección de los datos

Dado que las particiones del conjunto original fueron exportadas a archivos independientes, se aplicó un muestreo aleatorio simple del 10% dentro de cada partición por separado. Esto asegura que cada subconjunto mantenga su proporción interna, evitando introducir sesgos al momento de construir el conjunto M. Posteriormente, se unieron todas las muestras para formar un único conjunto que será utilizado en la etapa de modelado.

In [113]:
# Ruta
ruta_archivo = "/content/drive/MyDrive/MNA/Analisis de grandes volumenes de datos/Data/particiones"

# Cargar cada partición
p1 = spark.read.csv(f"{ruta_archivo}/output_ETHBTC_M5.csv/", header=True, inferSchema=True)
p2 = spark.read.csv(f"{ruta_archivo}/output_BTCUSDT_M15.csv/", header=True, inferSchema=True)
p3 = spark.read.csv(f"{ruta_archivo}/output_BNBUSDT_H1.csv/", header=True, inferSchema=True)
p4 = spark.read.csv(f"{ruta_archivo}/output_ETHUSDT_M30.csv/", header=True, inferSchema=True)

# Aplicar muestreo aleatorio simple del 10% en cada partición
m1 = p1.sample(withReplacement=False, fraction=0.1, seed=42)
m2 = p2.sample(withReplacement=False, fraction=0.1, seed=42)
m3 = p3.sample(withReplacement=False, fraction=0.1, seed=42)
m4 = p4.sample(withReplacement=False, fraction=0.1, seed=42)

df_M = m1.union(m2).union(m3).union(m4)

# Verificar tamaño total
df_M.count()

109743

# **Pregunta - 3:** Preparación de los datos

In [114]:
df_M.printSchema()

root
 |-- datetime: timestamp (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: double (nullable = true)
 |-- coin: string (nullable = true)
 |-- frequency: string (nullable = true)



In [115]:
# Verificar valores nulos por columna
df_M.select([sum(col(c).isNull().cast("int")).alias(c) for c in df_M.columns]).show()


+--------+----+----+---+-----+------+----+---------+
|datetime|open|high|low|close|volume|coin|frequency|
+--------+----+----+---+-----+------+----+---------+
|       0|   0|   0|  0|    0|     0|   0|        0|
+--------+----+----+---+-----+------+----+---------+



In [116]:
# Transformar el datetime
df_M_clean = df_M.withColumn("hour", hour("datetime")) \
                 .withColumn("day_of_week", dayofweek("datetime"))

In [117]:
print(df_M_clean.columns)

['datetime', 'open', 'high', 'low', 'close', 'volume', 'coin', 'frequency', 'hour', 'day_of_week']


In [118]:
# String index a columnas que son String
indexer_coin = StringIndexer(inputCol="coin", outputCol="coin_index")
indexer_freq = StringIndexer(inputCol="frequency", outputCol="freq_index")

df_indexed = indexer_coin.fit(df_M_clean).transform(df_M_clean)
df_indexed = indexer_freq.fit(df_indexed).transform(df_indexed)

In [119]:
# Vector assembler
assembler = VectorAssembler(
    inputCols=["open", "high", "low", "close", "volume", "hour", "day_of_week", "coin_index", "freq_index"],
    outputCol="features"
)
df_vector = assembler.transform(df_indexed)

In [120]:
df_vector = df_vector.withColumn("label", when(col("close") > col("open"), 1).otherwise(0))

In [121]:
df_vector.show()

+-------------------+--------+--------+--------+--------+--------+------+---------+----+-----------+----------+----------+--------------------+-----+
|           datetime|    open|    high|     low|   close|  volume|  coin|frequency|hour|day_of_week|coin_index|freq_index|            features|label|
+-------------------+--------+--------+--------+--------+--------+------+---------+----+-----------+----------+----------+--------------------+-----+
|2020-04-04 11:10:00| 0.02089|0.020908|0.020879|0.020883| 171.026|ETHBTC|       M5|  11|          7|       0.0|       0.0|[0.02089,0.020908...|    0|
|2020-04-04 11:55:00|0.020847|0.020847|0.020824|0.020824| 292.658|ETHBTC|       M5|  11|          7|       0.0|       0.0|[0.020847,0.02084...|    0|
|2020-04-04 12:05:00|0.020819|0.020843|0.020813|0.020843| 276.336|ETHBTC|       M5|  12|          7|       0.0|       0.0|[0.020819,0.02084...|    1|
|2020-04-04 14:45:00|0.020955|0.020961|0.020949|0.020958| 933.385|ETHBTC|       M5|  14|          7|

In [122]:
df_M = df_vector;
df_M.select("features", "label").show(5, truncate=False)

+---------------------------------------------------------------+-----+
|features                                                       |label|
+---------------------------------------------------------------+-----+
|[0.02089,0.020908,0.020879,0.020883,171.026,11.0,7.0,0.0,0.0]  |0    |
|[0.020847,0.020847,0.020824,0.020824,292.658,11.0,7.0,0.0,0.0] |0    |
|[0.020819,0.020843,0.020813,0.020843,276.336,12.0,7.0,0.0,0.0] |1    |
|[0.020955,0.020961,0.020949,0.020958,933.385,14.0,7.0,0.0,0.0] |1    |
|[0.020966,0.021011,0.020957,0.020989,4019.397,15.0,7.0,0.0,0.0]|1    |
+---------------------------------------------------------------+-----+
only showing top 5 rows



# **Pregunta - 4:** Preparación del conjunto de entrenamiento y prueba

Para esta etapa se utilizó la técnica de muestreo aleatorio simple con división de 70/30, lo que significa que el 70% de los registros de la muestra M se destinaron al conjunto de entrenamiento y el 30% al conjunto de prueba. Esta estrategia es ampliamente utilizada en problemas de clasificación binaria, ya que permite disponer de suficientes ejemplos para entrenar el modelo sin dejar de reservar una proporción representativa para su validación.

In [123]:
# 70% entrenamiento-30% prueba
train_data, test_data = df_M.randomSplit([0.7, 0.3], seed=42)

print("Tamaño del conjunto de entrenamiento:", train_data.count())
print("Tamaño del conjunto de prueba:", test_data.count())

Tamaño del conjunto de entrenamiento: 77053
Tamaño del conjunto de prueba: 32690


# **Pregunta - 5:** Construcción de modelos de aprendizaje supervisado y no supervisado

En esta etapa se implementaron dos tipos de modelos: uno de aprendizaje supervisado (GBTClassifier) y otro de aprendizaje no supervisado (KMeans) con el objetivo de identificar patrones y comportamientos en los datos financieros.

**APRENDIZAJE SUPERVISADO - GBTClassifier**

In [124]:
# Crear modelo
modelo_gbt = GBTClassifier(labelCol="label", featuresCol="features", maxIter=20)

# Entrenar
modelo_entrenado = modelo_gbt.fit(train_data)

# Predecir
predicciones = modelo_entrenado.transform(test_data)
predicciones.select("features", "label", "prediction", "probability").show(5, truncate=False)

+--------------------------------------------------------------+-----+----------+----------------------------------------+
|features                                                      |label|prediction|probability                             |
+--------------------------------------------------------------+-----+----------+----------------------------------------+
|[0.020819,0.020843,0.020813,0.020843,276.336,12.0,7.0,0.0,0.0]|1    |0.0       |[0.5185261079290094,0.48147389207099056]|
|[0.020929,0.020949,0.020915,0.020941,1380.09,16.0,7.0,0.0,0.0]|1    |0.0       |[0.5186203459731216,0.48137965402687843]|
|[0.020877,0.020896,0.020867,0.020872,406.187,17.0,7.0,0.0,0.0]|0    |0.0       |[0.5186203459731216,0.48137965402687843]|
|[0.020998,0.021029,0.020993,0.021021,394.687,17.0,7.0,0.0,0.0]|1    |0.0       |[0.5186203459731216,0.48137965402687843]|
|[0.021007,0.021027,0.021007,0.021026,245.233,19.0,7.0,0.0,0.0]|1    |1.0       |[0.48925305413882636,0.5107469458611736]|
+---------------

In [125]:
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predicciones)
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.5142


Se utilizó el modelo GBTClassifier para predecir si el precio subiría durante un periodo determinado, utilizando como variable objetivo (label) una comparación entre el precio de cierre y el de apertura. A pesar de entrenarse con un conjunto de características relevantes (precios, volumen, horario, codificaciones), el modelo alcanzó una precisión (accuracy) de solo 0.5119, lo que indica que su desempeño fue cercano al azar. Esto sugiere que las características utilizadas no fueron lo suficientemente representativas o que la relación entre variables es demasiado compleja para ser capturada por este modelo en su configuración actual.

**APRENDIZAJE NO SUPERVISADO - KMeans**

In [126]:
# Crear modelo
kmeans = KMeans(featuresCol="features", predictionCol="cluster", k=3, seed=42)

# Entrenar
modelo_kmeans = kmeans.fit(train_data)

# Predecir
df_clusterizado = modelo_kmeans.transform(test_data)

In [127]:
# Evaluar la calidad del agrupamiento
evaluator = ClusteringEvaluator(predictionCol="cluster")

silhouette = evaluator.evaluate(df_clusterizado)
print("Silhouette Score:", silhouette)

Silhouette Score: 0.9122222066804248


In [128]:
df_clusterizado.select("features", "cluster").show(10, truncate=False)

+--------------------------------------------------------------+-------+
|features                                                      |cluster|
+--------------------------------------------------------------+-------+
|[0.020819,0.020843,0.020813,0.020843,276.336,12.0,7.0,0.0,0.0]|0      |
|[0.020929,0.020949,0.020915,0.020941,1380.09,16.0,7.0,0.0,0.0]|0      |
|[0.020877,0.020896,0.020867,0.020872,406.187,17.0,7.0,0.0,0.0]|0      |
|[0.020998,0.021029,0.020993,0.021021,394.687,17.0,7.0,0.0,0.0]|0      |
|[0.021007,0.021027,0.021007,0.021026,245.233,19.0,7.0,0.0,0.0]|0      |
|[0.021034,0.021047,0.021021,0.021033,343.36,19.0,7.0,0.0,0.0] |0      |
|[0.021097,0.0211,0.02109,0.021097,205.75,20.0,7.0,0.0,0.0]    |0      |
|[0.021151,0.021154,0.021141,0.021144,258.538,1.0,1.0,0.0,0.0] |0      |
|[0.021173,0.021195,0.021168,0.021192,492.898,2.0,1.0,0.0,0.0] |0      |
|[0.020941,0.020984,0.020941,0.020983,191.958,3.0,1.0,0.0,0.0] |0      |
+--------------------------------------------------

In [129]:
# Caracteristicas por clusters
df_clusterizado.groupBy("cluster").avg("volume", "close", "open", "high", "low").show()

+-------+------------------+------------------+------------------+------------------+------------------+
|cluster|       avg(volume)|        avg(close)|         avg(open)|         avg(high)|          avg(low)|
+-------+------------------+------------------+------------------+------------------+------------------+
|      0|3297.1980793132648| 1234.698137659387|1234.5113043940396| 1238.231915706297| 1230.778704361135|
|      1|  954.243077316049| 36793.99846296302| 36797.88346296302|36883.385839506205|36705.808419753135|
|      2|177281.45242781658|123.88940393013095|123.92187445414848|126.12014388646286|120.63212598253278|
+-------+------------------+------------------+------------------+------------------+------------------+



Se aplicó el algoritmo KMeans con k=3 clústeres para segmentar los registros del mercado en grupos según su similitud en variables numéricas. El modelo generó clústeres con un Silhouette Score de 0.91, lo que refleja una excelente calidad de agrupamiento. El análisis por grupo mostró diferencias claras en características como volumen y precios, permitiendo identificar tres comportamientos distintos del mercado.